# Module 307 — LLMPics Skills Assessment

This notebook is a scoped field guide for the HTB Academy LLMPics Skills Assessment. HTB says to interact with the web app directly in a browser; SSH port forwarding is not needed. The target host and port change between instances, so enter the current assignment below.

The notebook is intentionally lightweight and uses only Python's standard library. It does not train a model, submit an Academy answer, brute-force keys, or store the test password, a recovered key, or the flag.

## 1. Scope the current target

Copy the IP and port from the currently spawned HTB target. Do not reuse a previous target. The empty values below prevent accidental requests to a stale or unintended host. Only the known LLMPics routes listed here are accepted by the read-only helper.

In [ ]:
import ipaddress

# Enter the current HTB-assigned target; do not commit these temporary values.
TARGET_HOST = ''  # Example shape: '192.0.2.10'
TARGET_PORT = None  # Example shape: 12345

KNOWN_PATHS = {'/', '/imagebot', '/adminbot', '/login', '/profile'}

def build_base_url(host, port):
    if not host or type(port) is not int or not 1 <= port <= 65535:
        raise ValueError('Enter the current target IP and a valid port first.')
    address = ipaddress.ip_address(host)
    return f'http://{address}:{port}'

BASE_URL = build_base_url(TARGET_HOST, TARGET_PORT) if TARGET_HOST and TARGET_PORT else None
print('Scoped target ready.' if BASE_URL else 'Set TARGET_HOST and TARGET_PORT before making a request.')

## 2. Read-only page checks (optional)

The following helper sends one ordinary GET request to a known page, such as the home page or Adminbot without a key. It does not log in, submit chat messages, or perform an action. Use the browser for the assessment's interactive work. An HTTP error still may include a useful access-control response.

In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def inspect_known_page(path, timeout_seconds=5):
    if BASE_URL is None:
        raise RuntimeError('Set the current HTB target first.')
    if path not in KNOWN_PATHS:
        raise ValueError('Path is outside the notebook allowlist.')

    request = Request(BASE_URL + path, headers={'User-Agent': 'HTB-Academy-Study-Notebook'})
    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            status = response.status
            body = response.read(3000).decode('utf-8', errors='replace')
    except HTTPError as error:
        status = error.code
        body = error.read(3000).decode('utf-8', errors='replace')
    except URLError as error:
        print('Target is not reachable from this Python environment:', error.reason)
        return None

    print('HTTP status:', status)
    print(body[:1000])
    return {'path': path, 'status': status, 'body_preview': body[:1000]}

# Example read-only checks; uncomment only when you want to inspect these pages.
# home_result = inspect_known_page('/')
# adminbot_result = inspect_known_page('/adminbot')

## 3. What is known so far

Record observations, not assumptions. Current observations from the spawned lab: the home page links to Imagebot, Adminbot, and Login; Imagebot can return a random photo and supports keyword search; Adminbot responds that an `admin_key` is required, and an invalid test value is rejected; a benign Imagebot question worked; a direct request for hidden configuration was refused. These observations do not reveal the key or prove which backend technique is vulnerable.

The HTB page provides a test username and password. Enter the password only in the lab's login form; do not write it into notebook cells or outputs. The active target IP is also temporary and must not be committed.

## 4. Methodical investigation checklist

1. Confirm the current target is reachable and inspect only the visible routes and features.
2. Establish a benign Imagebot baseline; record whether it returns text, a selected image, or an error.
3. Use the HTB test account in the browser if a feature requires authentication. Keep the password out of the notebook.
4. Inspect Adminbot's access-control behavior without guessing or brute-forcing keys.
5. Select one course hypothesis based on evidence: prompt injection, unsafe HTML/Markdown rendering, SQL handling, command/tool execution, or excessive agency.
6. Change one input at a time and observe the actual page or side effect. A model's claim alone is not proof.
7. Verify the requested outcome in LLMPics and then submit the flag through the Academy page.

Keep all activity scoped to the active HTB instance. Do not store a recovered key, personal data, or the flag in this notebook or Git.

## 5. Redacted observation log

Use short summaries. Never paste a password, admin key, session cookie, private user data, or flag. The log is in memory only; it is not automatically saved to disk.

In [ ]:
observations = []

def record_observation(stage, action, redacted_result, verified=False):
    item = {
        'stage': stage,
        'action': action,
        'result_summary': redacted_result,
        'verified_in_application': bool(verified),
    }
    observations.append(item)
    return item

# Example: record only the access-control behavior, not any secret value.
record_observation('recon', 'open Adminbot without a key', 'page requests admin_key', verified=True)
observations

## COAE review

Trace user input → model or tool → output sink/action → authorization check. Distinguish prompt injection from insecure output handling, excessive agency, and flaws inside a downstream function. Select mitigations for the actual boundary, and verify impact in the application. The assessment is complete only when HTB confirms the answer.